# Deploy DestinationExecutor

This notebook deploys `DestinationExecutor.sol` to Base or Arbitrum.

## Prereqs
- Use the `myvenv` kernel.
- Export the environment variable:
  - `DEPLOYER_PRIVATE_KEY`
- Provide the relayer address via `RELAYER_ADDRESS` or `RELAYER_PRIVATE_KEY`.
- Set the chain and addresses directly in the notebook.
- The deployer wallet must have gas on the target chain.


In [10]:
import os
from pathlib import Path
from dotenv import load_dotenv
from solcx import compile_standard, install_solc
from web3 import Web3

backend_env = Path("..") / "backend" / ".env"
if backend_env.exists():
    load_dotenv(backend_env)
else:
    load_dotenv(Path("..") / ".env")

# CHAIN = "base"  # base | arbitrum
CHAIN = "arbitrum"  # base | arbitrum

CHAIN_CONFIG = {
    "base": {
        "chain_id": 8453,
        "rpc": "https://base-rpc.publicnode.com",
        "usdc": "0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913",
    },
    "arbitrum": {
        "chain_id": 42161,
        "rpc": "https://arb1.arbitrum.io/rpc",
        "usdc": "0xaf88d065e77c8cC2239327C5EDb3A432268e5831",
    },
}


def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise ValueError(f"Missing env var: {name}")
    return value


def require_address(label: str, value: str) -> str:
    if not value or value == "0x0000000000000000000000000000000000000000":
        raise ValueError(f"Set {label} address")
    return Web3.to_checksum_address(value)


def resolve_relayer_address() -> str:
    explicit = os.getenv("RELAYER_ADDRESS")
    if explicit:
        return Web3.to_checksum_address(explicit)

    relayer_key = os.getenv("RELAYER_PRIVATE_KEY")
    if relayer_key:
        relayer_account = Web3().eth.account.from_key(relayer_key)
        return Web3.to_checksum_address(relayer_account.address)

    deployer_key = os.getenv("DEPLOYER_PRIVATE_KEY")
    if deployer_key:
        deployer_account = Web3().eth.account.from_key(deployer_key)
        return Web3.to_checksum_address(deployer_account.address)

    raise ValueError("Set RELAYER_ADDRESS, RELAYER_PRIVATE_KEY, or DEPLOYER_PRIVATE_KEY")


config = CHAIN_CONFIG.get(CHAIN)
if not config:
    raise ValueError(f"Unsupported CHAIN: {CHAIN}")

CHAIN_ID = config["chain_id"]
rpc_url = config["rpc"]
private_key = require_env("DEPLOYER_PRIVATE_KEY")
relayer_address = resolve_relayer_address()
usdc_address = require_address("USDC_ADDRESS", config["usdc"])

print("Deploying to:", CHAIN)
print("USDC:", usdc_address)
print("Relayer:", relayer_address)


Deploying to: arbitrum
USDC: 0xaf88d065e77c8cC2239327C5EDb3A432268e5831
Relayer: 0x2299fFB140C4f437E0015fdd9e1956feb2471eA6


In [11]:
w3 = Web3(Web3.HTTPProvider(rpc_url))
if not w3.is_connected():
    raise ConnectionError("Failed to connect to RPC")

install_solc("0.8.19")
source = Path("DestinationExecutor.sol").read_text()

compiled = compile_standard(
    {
        "language": "Solidity",
        "sources": {"DestinationExecutor.sol": {"content": source}},
        "settings": {"outputSelection": {"*": {"*": ["abi", "evm.bytecode"]}}},
    },
    solc_version="0.8.19",
)

contract_data = compiled["contracts"]["DestinationExecutor.sol"]["DestinationExecutor"]
abi = contract_data["abi"]
bytecode = contract_data["evm"]["bytecode"]["object"]


In [12]:
account = w3.eth.account.from_key(private_key)
print("Deployer:", account.address)

contract = w3.eth.contract(abi=abi, bytecode=bytecode)
constructor = contract.constructor(usdc_address, relayer_address)

gas_estimate = constructor.estimate_gas({"from": account.address})
latest_block = w3.eth.get_block("latest")
base_fee = latest_block.get("baseFeePerGas")

tx_params = {
    "from": account.address,
    "nonce": w3.eth.get_transaction_count(account.address),
    "gas": int(gas_estimate * 1.2),
    "chainId": CHAIN_ID,
}
if base_fee is not None:
    priority_fee = w3.eth.max_priority_fee
    tx_params["maxPriorityFeePerGas"] = priority_fee
    tx_params["maxFeePerGas"] = base_fee * 2 + priority_fee
else:
    tx_params["gasPrice"] = w3.eth.gas_price

tx = constructor.build_transaction(tx_params)

signed = account.sign_transaction(tx)
tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
print("Tx hash:", tx_hash.hex())

receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
print("Contract address:", receipt.contractAddress)


Deployer: 0x2299fFB140C4f437E0015fdd9e1956feb2471eA6
Tx hash: 3852f56318e4f8e17cf5a29cc8ec399deff10929a8a25c26bf4662d7ae1ee5a7
Contract address: 0x77108D9f93478222697815Bb8DE1ddc659bC83A9
